# 02 - Exploratory Data Analysis

NYC Yellow Taxi Trip Duration Prediction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

OUTPUT_DIR = 'notebooks/images'

df = pd.read_parquet('data/processed/cleaned.parquet')
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst 5 rows:\n{df.head()}")

In [ ]:
# Basic stats
print(f"Date range: {df['tpep_pickup_datetime'].min()} to {df['tpep_pickup_datetime'].max()}")
print(f"Unique pickup locations: {df['PULocationID'].nunique()}")
print(f"Unique dropoff locations: {df['DOLocationID'].nunique()}")

## 2. Target Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8))

# Original distribution
axes[0].hist(df['trip_duration_seconds'], bins=50, edgecolor='black', alpha=0.7)
mean_val = df['trip_duration_seconds'].mean()
median_val = df['trip_duration_seconds'].median()
axes[0].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.0f}s')
axes[0].axvline(median_val, color='blue', linestyle='--', linewidth=2, label=f'Median: {median_val:.0f}s')
axes[0].set_xlabel('Trip Duration (seconds)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Trip Duration Distribution')
axes[0].legend()

# Log-transformed distribution
log_duration = np.log1p(df['trip_duration_seconds'])
axes[1].hist(log_duration, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].axvline(log_duration.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {log_duration.mean():.2f}')
axes[1].axvline(log_duration.median(), color='blue', linestyle='--', linewidth=2, label=f'Median: {log_duration.median():.2f}')
axes[1].set_xlabel('Log(Trip Duration + 1)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log-Transformed Trip Duration Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Missing Values

In [ ]:
missing = df.isnull().sum().to_frame('null_count')
missing['null_pct'] = (missing['null_count'] / len(df)) * 100
print(f"Missing values:\n{missing[missing['null_count'] > 0]}")

plt.figure(figsize=(12, 8))
sns.heatmap(df.isnull(), cbar=True, cmap='viridis', yticklabels=False)
plt.title('Missing Values Heatmap')
plt.xlabel('Columns')
plt.ylabel('Rows')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Distributions

In [ ]:
df['hour_of_day'] = df['tpep_pickup_datetime'].dt.hour

features = ['trip_distance', 'passenger_count', 'PULocationID', 'DOLocationID', 'VendorID', 'hour_of_day']
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for idx, feat in enumerate(features):
    ax = axes[idx // 3, idx % 3]
    ax.hist(df[feat].dropna(), bins=30, edgecolor='black', alpha=0.7)
    ax.set_xlabel(feat)
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of {feat}')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Matrix

In [ ]:
df['pickup_hour'] = df['hour_of_day']
df['pickup_dayofweek'] = df['tpep_pickup_datetime'].dt.dayofweek

corr_features = ['trip_distance', 'passenger_count', 'VendorID', 'pickup_hour', 'pickup_dayofweek', 'trip_duration_seconds']
corr_df = df[corr_features].copy()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', 
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Features vs Target

In [ ]:
sample = df.sample(n=10000, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(12, 8))

# Scatter: trip_distance vs duration
axes[0].scatter(sample['trip_distance'], sample['trip_duration_seconds'], alpha=0.3, s=10)
axes[0].set_xlabel('Trip Distance')
axes[0].set_ylabel('Trip Duration (seconds)')
axes[0].set_title('Trip Distance vs Duration (sample 10k)')

# Box plot: pickup_hour vs duration
hourly_stats = df.groupby('pickup_hour')['trip_duration_seconds'].median().reset_index()
df.boxplot(column='trip_duration_seconds', by='pickup_hour', ax=axes[1])
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Trip Duration (seconds)')
axes[1].set_title('Duration by Hour')
plt.suptitle('')

# Box plot: day_of_week vs duration
df['day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek
df.boxplot(column='trip_duration_seconds', by='day_of_week', ax=axes[2])
axes[2].set_xlabel('Day of Week (0=Mon, 6=Sun)')
axes[2].set_ylabel('Trip Duration (seconds)')
axes[2].set_title('Duration by Day of Week')
plt.suptitle('')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/features_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Findings

- **Target distribution shape**: Trip duration is right-skewed with most trips under 30 minutes; log transformation normalizes the distribution
- **Strongest feature correlated with duration**: `trip_distance` shows the highest correlation with trip duration
- **Peak hours for long trips**: Longer trips tend to occur during rush hours (morning and evening commute times)
- **Surprising patterns**: Location-based features (PULocationID, DOLocationID) may have strong predictive power due to traffic patterns
- **Modeling implication**: Consider log-transforming the target variable for regression; distance and time features should be primary predictors